# 9.3. Language Models
D2L의 Language Models장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [2]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 언어 모델(Language Model)이란?

앞에서 텍스트를 단어 또는 문자와 같은 토큰(token)의 시퀀스로 변환했다. 만약에 길이가 $T$인 문장이 다음과 같다고 하자.

$$
x_1, x_2, \dots, x_T
$$

언어 모델의 목표는 이 전체 토큰 시퀀스가 등장할 확률을 모델링하는 것이다.

$$
P(x_1, x_2, \dots, x_T)
$$

예를 들어서 `deep learning is fun`이라는 문장이 있을 때 언어 모델은 단순히 단어 하나하나를 따로 보는 것이 아니라, deep -> learning -> is -> fun 이라는 순서까지 폼함한 전체 문장의 확률을 다룬다.

언어 모델이 잘 학습되었다면 지금까지 나온 단어들을 보고 다음 단어가 무엇일지 예측할 수 있다.

$$
P(x_t \mid x_1,\dots,x_{t-1})
$$

`지금까지의 토큰 -> 다음 토큰 예측`이 언어 모델의 가장 핵심적인 역할이다.

## 2. 문장의 확률을 조건부 확률로 분해하기

확률의 Chain Rule을 사용하면 전체 문장의 확률을 다음처럼 분해할 수 있다.

$$
\prod_{t=1}^{T}
P(x_t \mid x_1,\dots,x_{t-1})
$$

예를 들어 deep learning is fun 이면
$$
P(\text{deep},\text{learning},\text{is},\text{fun})
$$

이렇게 계산할 수 있다.
$$
P(\text{deep})\times P(\text{learning}\mid\text{deep})\times P(\text{is}\mid\text{deep},\text{learning})\times P(\text{fun}\mid\text{deep},\text{learning},\text{is})
$$

문장의 확률은 이렇게 표현할 수 있다.
    첫 단어가 나올 확률 x 앞 단어를 봤을때 다음 단어 나올 확률 x ...

이 아이더이가 나중에 RNN과 Transformer의 언어 모델링으로 그대로 연결된다.

## 3. Markov 가정

문제가 하나 있는데 다음 단어를 예측할 때 이전의 모든 단어를 고려하면 계산이 매우 복잡해진다.

예를 들어서

$$
P(x_t\mid x_1,x_2,\dots,x_{t-1})
$$

이걸 매번 계산해야한다. 그래서 과거의 모든 정보를 사용하는 대신 최근 일부 정보만 사용하자고 가정할 수 있다. 이것을 Markov 가정이라고 한다.

예를 들어 1차 Markov 모델은 이렇게 가정한다.
$$
P(x_{t+1}\mid x_1,\dots,x_t)
\approx
P(x_{t+1}\mid x_t)
$$

아주 오래전 단어들은 무시하고 바로 이전 단어만 보고 다음 단어를 예측한다. 실제 언어에서는 먼 과거의 단어도 중요하기 때문에 이것은 상당히 강한 가정이다.

## 4. n-gram

Markov 가정을 이용한 대표적인 언어 모델이 n-gram 모델이다.

### Unigram

각 단어를 독립적으로 본다. 앞 단어를 전혀 고려하지 않는다.

$$
P(x_1,x_2,x_3,x_4)
\approx
P(x_1)P(x_2)P(x_3)P(x_4)
$$

### Bigram

바로 이전 단어 하나를 사용한다. 예를 들어서

    deep -> learning

이런 두 단어의 관계를 학습한다.

$$
P(x_1,x_2,x_3,x_4)
\approx
P(x_1)
P(x_2\mid x_1)
P(x_3\mid x_2)
P(x_4\mid x_3)
$$

### Trigram

이전 단어 두 개를 사용한다. 예를 들어

    deep learning -> is

와 같은 관계를 학습한다.

$$
P(x_t\mid x_{t-2},x_{t-1})
$$

```text
Unigram  : 현재 단어만
Bigram   : 이전 1개 단어 사용
Trigram  : 이전 2개 단어 사용
4-gram   : 이전 3개 단어 사용
```

n이 커질수록 더 긴 문맥을 볼 수 있다. 하지만 n이 커질수록 새로운 문제가 발생한다.

## 5. 단어 빈도로 확률 계산하기

전통적인 n-gram 언어 모델에서는 대규모 텍스트에서 단어가 몇 번 등장했는지를 세어 확률을 추정할 수 있다.

예를 들어서 `deep learning` 이라는 단어 조합의 확률은 대략

$$
\frac{
n(\text{deep, learning})
}{
n(\text{deep})
}
$$

이걸로 계산할 수 있다. 여기서

$$
n(\text{deep})
$$

은 deep이 등장한 횟수이고,

$$
n(\text{deep, learning})
$$

은 deep learning이 연속해서 등장한 횟수이다.

예를 들어서 deep 등장 횟수가 1000이고 deep learning 등장 횟수 = 600이면 0.6이 된다.

## 6. n-gram의 데이터 희소성 문제

n-gram에서 가장 큰 문제는 n이 커질수록 가능한 단어 조합이 폭발적으로 증가하는 것이다. 

예를 들어서 어휘가 10,000개라면 가능한 bigram은 이론적으로 10,000^2이 된다. trigram은 10,000^3이다.

그래서 실제로 충분히 자연스러운 문장이라도 학습 데이터에서 한 번도 등장하지 않을 수 있다.

예를 들어 학습 데이터에서

    artificial intelligence researcher

라는 조합이 한 번도 등장하지 않았다면 단순 빈도 기반 모델은

$$
P(\text{artificial intelligence researcher})=0
$$

처럼 처리할 수도 있다. 하지만 실제로는 충분히 가능한 문장이다. 이 현상을 `데이터 희소성(data sparsity)`문제라고 생각하면 된다.

## 7. Laplace Smoothing

학습 데이터에서 한 번도 보지 못했다고 해서 확률을 바로 0으로 만들면 문제가 생긴다. 그래서 등장 횟수에 작은 값을 추가하는 smoothing 기법을 사용할 수 있다.

가장 단순한 방법 중 하나가 Laplace smoothing이다.

`실제 등장 횟수 + 아주 작은 가상의 등장 횟수`를 하는 것이다.

한 번도 본 적 없는 단어 조합을 확률 0 으로 주는 게 아니라 매우 작은 확률로 준다.

하지만 n-gram 자체에는 여전히 문제가 많다.

- n이 커질수록 저장해야 할 조합이 너무 많다.
- 희귀한 문장 조합이 지나치게 많다.
- 단어의 의미적 유사성을 제대로 활용하지 못한다.
- 매우 긴 문맥을 처리하기 어렵다.

예를 들어

```text
cat
feline
```

은 의미적으로 비슷하지만 단순 빈도 기반 모델에서는 서로 다른 토큰일 뿐이다. 이러한 한계 때문에 이후에는 신경망 기반 언어 모델을 사용하게 된다.

## 8. 언어 모델은 어떻게 평가할까?

좋은 언어 모델이라면 주어진 문맥 다음에 올 단어를 높은 확률로 예측해야 한다.

예를 들어서

    It is raining ...

다음에 outside 같은 자연스러운 단어에는 높은 확률을 주고, banana같은 이상한 단어엔 낮은 확률을 줘야 한다.

따라서 실제 정답 토큰에 모델이 얼마나 높은 확률을 부여했는지를 측정할 수 있다. 이를 위해 평균 Cross Entropy를 사용할 수 있다.

$$
\frac{1}{n}
\sum_{t=1}^{n}
-\log
P(x_t\mid x_1,\dots,x_{t-1})
$$

모델이 실제 정답에 높은 확률을 주면 loss가 작아진다.

    정답 확률 ↑ -> -log(P) ↓ -> Cross Entropy ↓ -> 좋은 언어 모델

## 9. PerPlexity

언어 모델에서는 Cross Entropy를 그대로 사용하기도 하지만 Perplexity(PPL) 라는 지표를 많이 사용한다.

Perplexity는 평균 Cross Entropy에 exponential을 취한 값이다.
$$
\exp
\left(
-\frac{1}{n}
\sum_{t=1}^{n}
\log P(x_t\mid x_1,\dots,x_{t-1})
\right)
$$

`모델이 다음 단어를 고를 때 평균적으로 몇 개의 후보 사이에서 고민하고 있는가?` 정도로 이해하면 될 것이다.

예를 들어 PPL = 1 이면 거의 완벽하게 다음 토큰을 알고 있다는 뜻이다.

반대로 PPL = 100 이면 다음 토큰을 결정할 때 대략 100개의 후보 사이에서 헷갈리는 것과 비슷한 상태라고 생각할 수 있다.

따라서 기본적으로 Perplexity가 낮을수록 좋다.

$$
P(\text{정답})=1
$$

이걸 항상 예측할 수 있는 완벽한 모델이라면 PPL은 1이다.

## 10. 언어 모델의 학습 데이터 만들기

RNN을 학습시키기 위해서는 긴 텍스트 전체를 한 번에 넣기보다는 일정 길이의 작은 sequence로 나눈다.

예를 들어서 원래 데이터가 `A B C D E F G H`라고 해보자.

num_steps = 4라면 입력을 `A B C D`로 만들 수 있다. 그런데 언어 모델의 목표는 다음 토큰 예측이다. 그래서 정답 Y는 한 칸 오른쪽으로 이동시킨다.

```text
X : A B C D
Y : B C D E
```

그러면 모델은 사실 다음 네 문제를 동시에 학습하는 것이다.

```text
A       → B
A B     → C
A B C   → D
A B C D → E
```

이 구조가 매우 중요하다. 앞으로 RNN 코드를 보면 계속

```python
X = array[:, :-1]
Y = array[:, 1:]
```

같은 구조가 등장하는 이유가 바로 이것이다.

## 11. TimeMachine 데이터 만들기

PyTorch 버전에서는 이렇게 sequence를 만들 수 있다.

In [ ]:
import torch 
from d2l import torch as d2l

@d2l.add_to_class(d2l.TimeMachine)
def __init__(self, batch_size, num_steps,
             num_train=10000, num_val=5000):

    super(d2l.TimeMachine, self).__init__()

    self.save_hyperparameters()

    corpus, self.vocab = self.build(self._download())

    array = torch.tensor([
        corpus[i:i + num_steps + 1]
        for i in range(len(corpus) - num_steps)
    ])
    # 예를 들어 array = [1, 2, 3, 4, 5]라면
    self.X = array[:, :-1] # X = [1, 2, 3, 4]
    self.Y = array[:, 1:]  # Y = [2, 3, 4, 5]


## 12. Minibatch 확인하기

In [4]:
data = d2l.TimeMachine(
    batch_size=2,
    num_steps=10
)

for X, Y in data.train_dataloader():
    print("X:", X)
    print("Y:", Y)
    break

X: tensor([[ 0,  2,  4,  4,  6, 17, 21,  0,  2, 15],
        [22,  0,  2, 19,  6,  0, 24, 19, 16, 15]])
Y: tensor([[ 2,  4,  4,  6, 17, 21,  0,  2, 15, 26],
        [ 0,  2, 19,  6,  0, 24, 19, 16, 15,  8]])


출력 형태는 이런 식이다.

```text
X: [a b c d e ...] [k l m n o ...] 
Y: [b c d e f ...] [l m n o p ...]

X.shape = [batch_size, num_steps] 
Y.shape = [batch_size, num_steps]

batch_size = 2
num_steps = 10

이면

X.shape = [2, 10]
Y.shape = [2, 10]
```

여기서 각 숫자는 문자나 단어 자체가 아니라 vocabulary에서 해당 토큰에 부여된 index이다.

## 13. 이번 장의 핵심 흐름

언어 모델을 한 문장으로 표현하면 이전 토큰들을 보고 다음 토큰의 확률 분포를 예측하는 모델이다.

확률적으로는 이것을 학습한다.

$$
P(x_t\mid x_1,\dots,x_{t-1})
$$

전체 문장 확률은 이렇게 표현한다.
$$
\prod_{t=1}^{T}
P(x_t\mid x_1,\dots,x_{t-1})
$$

초기의 방법은 n-gram이었다.

```text
Unigram
   ↓
Bigram
   ↓
Trigram
   ↓
더 긴 n-gram
```

하지만 긴 문맥으로 갈수록 데이터 희소성과 저장 공간 문제가 심각해진다. 그래서 앞으로는 아래 방식으로 발전하게 된다.

```text
단순 빈도 기반 n-gram
        ↓
Neural Language Model
        ↓
RNN
        ↓
LSTM / GRU
        ↓
Transformer
```

## 14. 오늘의 정리

- 언어 모델은 토큰 시퀀스의 확률을 모델링한다.
- 핵심 문제는 이전 토큰을 이용해 다음 토큰을 예측하는 것이다.
- 전체 문장 확률은 조건부 확률의 곱으로 분해할 수 있다.
- n-gram은 최근 몇 개의 토큰만 이용해 다음 토큰을 예측한다.
- n이 커질수록 문맥은 많이 볼 수 있지만 데이터 희소성 문제가 심해진다.
- Laplace smoothing은 보지 못한 조합의 확률이 0이 되는 문제를 완화한다.
- 신경망 언어 모델은 n-gram의 여러 한계를 해결하기 위해 사용된다.
- 언어 모델의 대표적인 평가 지표가 Perplexity(PPL)이다.
- PPL은 낮을수록 좋은 모델이며 최솟값은 1이다.
- 언어 모델의 학습 데이터에서는 입력 X에 대해 정답 Y를 한 토큰 앞으로 이동시킨다.
- 즉 X = [A, B, C, D]라면 Y = [B, C, D, E]가 된다.
- 이 X/Y 구조를 이해해야 다음 장의 RNN 학습 코드를 제대로 이해할 수 있다.